In [2]:
import numpy as np
from networkx.algorithms.operators.product import tensor_product


def all_close(a_, b_, atol=1e-10, rtol=1e-10):
    # --- compare per level ---
    for k in range(depth + 1):

        x = a_[k]
        y = b_[k]
        if x is None:
            x = np.zeros_like(y)
        delta = x - y
        print(f"level {k} abs: ", abs(delta).max() < atol) if hasattr(delta, "max") else abs(delta) < atol
        rel_dist = (abs(delta) / abs(x)).max() if hasattr(delta, "max") else abs(delta) / abs(x)
        print(f"level {k} rel: ", rel_dist < rtol if rel_dist < rtol else rel_dist)


B, L, d, depth = 10000, 101, 3, 7

np.random.seed(12335)
path_a = np.array(np.random.randn(B, L - 1, d), dtype=np.float64) * (1.0 / (L - 1)) ** 2
path_a = np.concat([np.zeros((B, 1, d)), np.cumsum(path_a, axis=1)], axis=1)

np.random.seed(23466)
path_b = np.array(np.random.randn(B, L - 1, d), dtype=np.float64) * (1.0 / (L - 1)) ** 2
path_b = np.concat([path_a[:, -1:, :], path_b], axis=1)
path_b = np.cumsum(path_b, axis=1)
path_c = np.concatenate([path_a, path_b[:, 1:]], axis=1)

In [1]:
from tensordev.src.tensordev.core import Universal

import numpy as np

NumpyCore = Universal(np)
B, L, d, depth = 10000, 101, 3, 7
path_a = np.concat([np.zeros((B, 1, d)), np.cumsum(np.array(np.random.randn(B, L - 1, d)) * (1.0 / (L - 1)) ** 2, axis=1)],
axis=1)


In [11]:
sig_a = NumpyCore.tensor_path_signature(path_a, trunc=depth)
len(sig_a), [lvl.shape for lvl in sig_a]

(8,
 [(10000, 1),
  (10000, 3),
  (10000, 9),
  (10000, 27),
  (10000, 81),
  (10000, 243),
  (10000, 729),
  (10000, 2187)])

In [12]:
sig_a = NumpyCore.tensor_path_signature(path_a, trunc=depth)
len(sig_a), [lvl.shape for lvl in sig_a]

path_b = np.array(np.random.randn(B, L - 1, d), dtype=np.float64) * (1.0 / (L - 1)) ** 2
path_b = np.concat([path_a[:, -1:, :], path_b], axis=1)
path_b = np.cumsum(path_b, axis=1)
path_c = np.concatenate([path_a, path_b[:, 1:]], axis=1)

sig_b = NumpyCore.tensor_development([path_b], trunc=depth)
sig_c = NumpyCore.tensor_development([path_c], trunc=depth, block_size=100)

sig_c_ = NumpyCore.tensor_product(sig_a, sig_b, trunc=depth)

for i in range(depth):
    assert np.allclose(sig_a[i], sig_c[i][:, 0]) # check if first block gives sig_a
    assert np.allclose(sig_c_[i], sig_c[i][:, 1]) # check if second block gives sig_a \otimes sig_b

In [2]:
from tensordev.src.tensordev.core import Jax
JaxCore = Jax()

sig_a = JaxCore.tensor_path_signature(path_a, trunc=depth)

In [5]:
path_b = np.array(np.random.randn(B, L - 1, d), dtype=np.float64) * (1.0 / (L - 1)) ** 2
path_b = np.concat([path_a[:, -1:, :], path_b], axis=1)
path_b = np.cumsum(path_b, axis=1)
path_c = np.concatenate([path_a, path_b[:, 1:]], axis=1)

sig_b = JaxCore.tensor_development([path_b], trunc=depth)
sig_c = JaxCore.tensor_development([path_c], trunc=depth, block_size=100)

sig_c_ = JaxCore.tensor_product(sig_a, sig_b, trunc=depth)

for i in range(depth):
    assert np.allclose(sig_a[i], sig_c[i][:, 0]) # check if first block gives sig_a
    assert np.allclose(sig_c_[i], sig_c[i][:, 1]) # check if second block gives sig_a \otimes sig_b

In [3]:
from array_api_compat import array_namespace

from tensordev.src.tensordev.core.einsum import Einsum
from tensordev.src.tensordev.core.jax import Jax as JaxClass
#from tensordev.src.tensordev.core.torch import Torch as TorchClass
from tensordev.src.tensordev.core.numba import Numba as NumbaClass

Numpy = Einsum(array_namespace(np.asarray(1)))
#Torch = Einsum(array_namespace(torch.empty(1)))
Numba = NumbaClass()
Jax = JaxClass()
#orch = TorchClass()

In [4]:
a = Numpy.tensor_development([path_a], trunc=depth)
b = Numpy.tensor_development([path_b], trunc=depth)
c = Numpy.tensor_development([path_c], trunc=depth, block_size=100)

c_ = Numpy.tensor_product(a, b, trunc=depth)

all_close(c_, [c_k[:, 1] for c_k in c])
all_close(a, [c_k[:, 0] for c_k in c])

level 0 abs:  True
level 0 rel:  True
level 1 abs:  True
level 1 rel:  True
level 2 abs:  True
level 2 rel:  True
level 3 abs:  True
level 3 rel:  True
level 4 abs:  True
level 4 rel:  True
level 5 abs:  True
level 5 rel:  True
level 6 abs:  True
level 6 rel:  True
level 7 abs:  True
level 7 rel:  True
level 0 abs:  True
level 0 rel:  True
level 1 abs:  True
level 1 rel:  True
level 2 abs:  True
level 2 rel:  True
level 3 abs:  True
level 3 rel:  True
level 4 abs:  True
level 4 rel:  True
level 5 abs:  True
level 5 rel:  True
level 6 abs:  True
level 6 rel:  True
level 7 abs:  True
level 7 rel:  True


In [5]:
a = Numba.tensor_development([path_a], trunc=depth)
b = Numba.tensor_development([path_b], trunc=depth)
c = Numba.tensor_development([path_c], trunc=depth, block_size=100)

c_ = Numba.tensor_product(a, b, trunc=depth)

all_close(c_, [c_k[:, 1] for c_k in c])
all_close(a, [c_k[:, 0] for c_k in c])

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
- Resolution failure for literal arguments:
reshape() supports contiguous array only
- Resolution failure for non-literal arguments:
None

During: resolving callee type: BoundFunction(array.reshape for array(float64, 2d, A))
During: typing of call at /Users/paulhager/PycharmProjects/levyMMD/tensordev/src/tensordev/core/numba.py (27)

File "src/tensordev/core/numba.py", line 27:
def tensor_product_homogeneous_numba(Ai, Bj):
    <source elided>
    Ai2 = Ai.reshape(B, i)
    Bj2 = Bj.reshape(B, j)
    ^

During: Pass nopython_type_inference

In [6]:
a = Jax.tensor_development([path_a], trunc=depth)
b = Jax.tensor_development([path_b], trunc=depth)
c = Jax.tensor_development([path_c], trunc=depth, block_size=100)

c_ = Jax.tensor_product(a, b, trunc=depth)

all_close(c_, [c_k[:, 1] for c_k in c])
all_close(a, [c_k[:, 0] for c_k in c])

level 0 abs:  True
level 0 rel:  True
level 1 abs:  True
level 1 rel:  True
level 2 abs:  True
level 2 rel:  True
level 3 abs:  True
level 3 rel:  True
level 4 abs:  True
level 4 rel:  True
level 5 abs:  True
level 5 rel:  True
level 6 abs:  True
level 6 rel:  True
level 7 abs:  True
level 7 rel:  True
level 0 abs:  True
level 0 rel:  True
level 1 abs:  True
level 1 rel:  True
level 2 abs:  True
level 2 rel:  True
level 3 abs:  True
level 3 rel:  True
level 4 abs:  True
level 4 rel:  True
level 5 abs:  True
level 5 rel:  True
level 6 abs:  True
level 6 rel:  True
level 7 abs:  True
level 7 rel:  True


In [9]:
import torch, sys
print("versions:", torch.__version__, sys.version)
@torch.compile
def f(x): return (x + 1).relu()
print(f(torch.randn(4,5)).shape)

versions: 2.9.0 3.13.7 | packaged by Anaconda, Inc. | (main, Sep  9 2025, 19:54:17) [Clang 17.0.6 ]
torch.Size([4, 5])


In [4]:
Torch.tensor_development([torch.from_numpy(path_a[:2,:2])], trunc=4)

W1025 17:30:03.598000 56068 site-packages/torch/fx/experimental/symbolic_shapes.py:6833] [0/5] _maybe_guard_rel() was called on non-relation expression Eq(s36, 1) | Eq(s36, s95)
W1025 17:30:03.690000 56068 site-packages/torch/fx/experimental/symbolic_shapes.py:6833] [0/6] _maybe_guard_rel() was called on non-relation expression Eq(s36, 1) | Eq(s36, s95)


(tensor([[1.],
         [1.]], dtype=torch.float64),
 tensor([[ 1.1758e-04, -2.5363e-05,  9.9509e-05],
         [-7.1629e-05, -8.2269e-05,  8.8346e-05]], dtype=torch.float64),
 tensor([[ 6.9120e-09, -1.4910e-09,  5.8499e-09, -1.4910e-09,  3.2164e-10,
          -1.2619e-09,  5.8499e-09, -1.2619e-09,  4.9510e-09],
         [ 2.5653e-09,  2.9464e-09, -3.1641e-09,  2.9464e-09,  3.3841e-09,
          -3.6341e-09, -3.1641e-09, -3.6341e-09,  3.9025e-09]],
        dtype=torch.float64),
 tensor([[ 2.7089e-13, -5.8436e-14,  2.2927e-13, -5.8436e-14,  1.2606e-14,
          -4.9457e-14,  2.2927e-13, -4.9457e-14,  1.9404e-13, -5.8436e-14,
           1.2606e-14, -4.9457e-14,  1.2606e-14, -2.7193e-15,  1.0669e-14,
          -4.9457e-14,  1.0669e-14, -4.1857e-14,  2.2927e-13, -4.9457e-14,
           1.9404e-13, -4.9457e-14,  1.0669e-14, -4.1857e-14,  1.9404e-13,
          -4.1857e-14,  1.6422e-13],
         [-6.1251e-14, -7.0349e-14,  7.5546e-14, -7.0349e-14, -8.0799e-14,
           8.6768e-14,  7.5546

In [ ]:
Torch.tensor_development([torch.from_numpy(path_a[:2])], trunc=4)

In [3]:
Jax.tensor_development([path_a], trunc=depth)

(Array([[1.],
        [1.],
        [1.],
        ...,
        [1.],
        [1.],
        [1.]], dtype=float32),
 Array([[-0.00111707, -0.00036911,  0.00222117],
        [-0.00103812, -0.00146729,  0.00205746],
        [-0.00077912, -0.00030637,  0.00038567],
        ...,
        [-0.0002506 ,  0.00053365, -0.00128545],
        [ 0.00026426, -0.00098056, -0.00140893],
        [-0.00034203,  0.00224007,  0.00088579]], dtype=float32),
 Array([[ 5.5758778e-07,  2.5502260e-07, -6.8514612e-07, ...,
         -5.7185912e-07, -2.3658708e-07,  1.4848393e-06],
        [ 5.8799094e-07,  2.9427437e-07, -5.3460070e-07, ...,
         -5.7165227e-07, -6.4517849e-07,  1.3434158e-06],
        [ 3.9856181e-07,  3.7165867e-07,  2.9605511e-07, ...,
         -4.4940660e-07, -9.0111911e-08,  3.8450187e-07],
        ...,
        [ 2.4250980e-07,  3.9071701e-08, -1.8034719e-07, ...,
          2.7005891e-07, -4.4100295e-07,  6.0728024e-07],
        [ 2.4985945e-07, -9.3559656e-08, -1.5825592e-07, ...,
       

In [7]:
Numpy.tensor_development([path_a], trunc=depth)

(array([[1.],
        [1.],
        [1.],
        ...,
        [1.],
        [1.],
        [1.]], shape=(10000, 1)),
 array([[-0.00111707, -0.00036911,  0.00222117],
        [-0.00103812, -0.00146729,  0.00205746],
        [-0.00077912, -0.00030637,  0.00038567],
        ...,
        [-0.0002506 ,  0.00053365, -0.00128545],
        [ 0.00026426, -0.00098056, -0.00140893],
        [-0.00034203,  0.00224007,  0.00088579]], shape=(10000, 3)),
 array([[ 5.57587608e-07,  2.55022522e-07, -6.85146335e-07, ...,
         -5.71858978e-07, -2.36587011e-07,  1.48483889e-06],
        [ 5.87990852e-07,  2.94274392e-07, -5.34600745e-07, ...,
         -5.71652284e-07, -6.45178494e-07,  1.34341550e-06],
        [ 3.98561820e-07,  3.71658664e-07,  2.96055113e-07, ...,
         -4.49406632e-07, -9.01118890e-08,  3.84501867e-07],
        ...,
        [ 2.42509847e-07,  3.90716737e-08, -1.80347160e-07, ...,
          2.70058953e-07, -4.41002850e-07,  6.07280192e-07],
        [ 2.49859434e-07, -9.35596348e-

In [2]:
from tensordev.src.tensordev.numpy import tensor_abra

In [ ]:
'